# Halts Root Cause Audit Notebook

Notebook metodol?gico de `halts v2`.

Objetivos:

- leer solo artefactos de `cache_v2`
- fijar la foto real del dataset por source
- abrir la calidad de evento y la taxonom?a de uso
- cerrar el `multisource_row_mismatch` con el matiz correcto
- dejar preparada la siguiente fase de cruce causal con `quotes` y `trades`


In [ ]:
from pathlib import Path
import runpy
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import Markdown, display, clear_output

sns.set_theme(style='whitegrid')
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 2000)
pd.set_option('display.max_colwidth', 200)

SCRIPT_00 = Path(r'C:/TSIS_Data/02_backtest_SmallCaps/notebooks/00_data_certification/auditoria/halts/cell_code/00_load_halts_audit_artifacts.py')
mod00 = runpy.run_path(str(SCRIPT_00))
CACHE_DIR = mod00['DEFAULT_CACHE_DIR']
manifest = mod00['load_manifest'](CACHE_DIR)
build_log = mod00['load_build_log'](CACHE_DIR)
arts = mod00['load_all_core_artifacts'](CACHE_DIR)
load_artifact = mod00['load_artifact']

display(Markdown(f"**cache_dir:** `{CACHE_DIR}`"))
display(pd.DataFrame([{
    'input_rows': manifest.get('input_rows'),
    'canonical_events': manifest.get('row_counts', {}).get('canonical_event_summary'),
    'warnings': '; '.join(manifest.get('warnings', [])),
    'duration_sec': build_log.get('duration_sec'),
}]))


## 1. Calidad por source

Primero fijamos si cada source est? utilizable como capa de verdad de evento o si arrastra un problema estructural de parseo o identidad.


In [ ]:
source_quality = arts['source_quality_summary']
display(source_quality)

plot_df = source_quality.melt(
    id_vars=['source', 'rows'],
    value_vars=['ticker_nonnull_rows', 'halt_date_nonnull_rows', 'halt_start_nonnull_rows', 'resume_trade_nonnull_rows'],
    var_name='metric',
    value_name='rows_value',
)
plt.figure(figsize=(12, 5))
sns.barplot(data=plot_df, x='source', y='rows_value', hue='metric')
plt.title('Completitud operativa por source')
plt.ylabel('rows')
plt.tight_layout()


In [ ]:
plot_pct = []
for metric in ['ticker_nonnull_rows', 'halt_date_nonnull_rows', 'halt_start_nonnull_rows', 'resume_trade_nonnull_rows']:
    tmp = source_quality[['source', 'rows', metric]].copy()
    tmp['metric'] = metric.replace('_nonnull_rows', '')
    tmp['pct'] = 100 * tmp[metric] / tmp['rows']
    plot_pct.append(tmp[['source', 'metric', 'pct']])
plot_pct = pd.concat(plot_pct, ignore_index=True)
display(plot_pct.pivot(index='source', columns='metric', values='pct').round(3))
plt.figure(figsize=(12, 5))
sns.barplot(data=plot_pct, x='source', y='pct', hue='metric')
plt.ylim(0, 105)
plt.title('% de completitud operativa por source')
plt.ylabel('% rows no nulas')
plt.tight_layout()


## 2. Completitud de campos cr?ticos

Aqu? miramos el contrato de campos. La pregunta es si el evento queda roto o si solo pierde granularidad en una parte concreta.


In [ ]:
field_completeness = arts['field_completeness_summary']
display(field_completeness.sort_values(['source', 'nonnull_pct'], ascending=[True, False]))

heat = field_completeness.pivot(index='field', columns='source', values='nonnull_pct')
plt.figure(figsize=(8, 6))
sns.heatmap(heat, annot=True, fmt='.1f', cmap='YlGnBu')
plt.title('% no nulo por campo y source')
plt.tight_layout()


## 3. Taxonom?a de uso real

No basta con contar filas. Aqu? se decide cu?nto del dataset sirve realmente para overlay y an?lisis causal intrad?a.


In [ ]:
tax = arts['event_taxonomy_summary']
display(tax)

tax = tax.copy()
tax['events_pct'] = 100 * tax['events'] / tax['events'].sum()
display(tax[['event_taxonomy', 'events', 'events_pct', 'source_rows', 'tickers']].round({'events_pct': 4}))
plt.figure(figsize=(10, 4.5))
sns.barplot(data=tax, y='event_taxonomy', x='events', color='#4C78A8')
plt.title('Taxonom?a de eventos de halt')
plt.tight_layout()


## 4. Din?mica temporal

Adem?s del estado estructural, interesa ver si la actividad de `halts` tiene concentraci?n temporal real o si el dataset parece dominado por ruido administrativo.


In [ ]:
canonical = arts['canonical_event_summary']
yearly = canonical.loc[canonical['halt_date'].notna()].copy()
yearly['year'] = yearly['halt_date'].dt.year.astype(int)
yearly = yearly.groupby('year', as_index=False).agg(events=('event_id_canonical', 'nunique'))
display(yearly.tail(20))
plt.figure(figsize=(12, 4.5))
sns.lineplot(data=yearly, x='year', y='events', marker='o', linewidth=2)
plt.title('Eventos can?nicos de halt por a?o')
plt.tight_layout()


## 5. Reconciliaci?n del builder multisource

Este bloque cierra el warning del `multisource_row_mismatch` con el matiz correcto: no es p?rdida opaca de cobertura, sino efecto de deduplicaci?n hist?rica de Nasdaq y de diferencias de normalizaci?n frente a nuestro builder enriquecido.


In [ ]:
recon = arts['multisource_builder_reconciliation']
display(recon)

plot_recon = recon.loc[recon['scope'] != 'persisted_multisource_parquet'].copy()
plt.figure(figsize=(10, 4))
ax = sns.barplot(data=plot_recon, x='scope', y='dedup_delta', color='#F28E2B')
plt.title('Dedup delta del builder multisource')
for i, row in plot_recon.reset_index(drop=True).iterrows():
    ax.text(i, row['dedup_delta'] + max(1, row['dedup_delta'] * 0.01), f"{int(row['dedup_delta'])}", ha='center', va='bottom', fontsize=9)
plt.tight_layout()


In [ ]:
source_sum = int(manifest.get('input_rows', 0))
persisted_rows = int(recon.loc[recon['scope'] == 'persisted_multisource_parquet', 'rows_post_builder_dedup'].iloc[0])
reconstructed_rows = int(recon.loc[recon['scope'] == 'all_sources_concat', 'rows_post_builder_dedup'].iloc[0])
display(pd.DataFrame([{
    'source_sum': source_sum,
    'persisted_multisource_rows': persisted_rows,
    'reconstructed_post_dedup_rows': reconstructed_rows,
    'source_sum_minus_persisted': source_sum - persisted_rows,
    'source_sum_minus_reconstructed': source_sum - reconstructed_rows,
    'reconstructed_minus_persisted': reconstructed_rows - persisted_rows,
}]))
display(Markdown('Lectura: el parquet hist?rico ya incorpora deduplicaci?n fuerte de Nasdaq. La peque?a diferencia entre reconstrucci?n actual y parquet persistido no apunta a eventos perdidos, sino a diferencias de normalizaci?n del builder enriquecido frente al builder hist?rico.'))


## 6. Overlap entre sources y residuo duro

Aqu? abrimos dos cosas distintas: el overlap laxo real entre `nasdaq` y `nyse`, y el residuo final que no se puede rescatar porque el raw ya viene vac?o.


In [ ]:
dup = arts['duplicate_analysis']
overlap = arts['cross_source_overlap_summary']
residual_rows = arts['nasdaq_final_residual_rows']

cross_overlap = overlap.loc[overlap['has_cross_source_overlap']].copy()
display(dup.head(30))
display(cross_overlap.head(30))
display(residual_rows)

overlap_bins = cross_overlap.groupby('source_count', as_index=False).agg(loose_keys=('loose_key', 'nunique'), rows=('rows', 'sum'))
display(overlap_bins)
plt.figure(figsize=(8, 4))
sns.barplot(data=overlap_bins, x='source_count', y='loose_keys', color='#76B7B2')
plt.title('Loose overlaps por n?mero de sources')
plt.xlabel('source_count')
plt.ylabel('loose keys')
plt.tight_layout()


In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(cross_overlap['rows'], bins=30, color='#59A14F')
plt.title('Distribuci?n del tama?o de overlap laxo entre sources')
plt.xlabel('rows por loose key')
plt.tight_layout()

bad_cases = canonical.loc[canonical['event_taxonomy'] == 'bad_unusable_event'].copy()
display(bad_cases)


## 7. Cobertura por ticker y concentraci?n

Este bloque separa actividad real de halt de simple presencia nominal de ticker en el dataset.


In [ ]:
coverage = arts['ticker_halt_coverage_summary']
year_presence = arts['ticker_year_halt_presence']
display(coverage.head(30))
display(year_presence.head(30))

top_cov = coverage.sort_values('halt_events_count', ascending=False).head(20)
plt.figure(figsize=(12, 5))
sns.barplot(data=top_cov, y='ticker', x='halt_events_count', color='#E45756')
plt.title('Top tickers por n?mero de halts')
plt.tight_layout()


In [ ]:
cov_nonzero = coverage.loc[coverage['halt_events_count'] > 0, 'halt_events_count'].copy()
display(cov_nonzero.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame('halt_events_count'))
plt.figure(figsize=(10, 4.5))
sns.histplot(cov_nonzero, bins=40, color='#B279A2', log_scale=(True, False))
plt.title('Distribuci?n de halts por ticker')
plt.xlabel('halt_events_count (escala log)')
plt.tight_layout()


## 8. Lectura ejecutiva de cierre de fase

La fase 1 queda cerrada as?:

- Nasdaq deja de ser un bloque roto y pasa a ser utilizable para ventana intrad?a.
- SEC sigue siendo verdad regulatoria, pero no timing intrad?a fino.
- El residuo duro se reduce a `11` raws vac?os de Nasdaq.
- El `multisource_row_mismatch` deja de interpretarse como p?rdida de cobertura.
- El siguiente trabajo ?til ya es causal: alinear `halts` con patrones observados en `quotes` y `trades`.


## 9. Primera lectura causal `halts -> quotes / trades`

Esta primera pasada no baja todav?a a intrad?a tick-a-tick. Lo que fija es la alineaci?n exhaustiva a nivel `ticker-date` para cuatro ventanas diarias alrededor del evento: `pre_halt_day`, `halt_day`, `resume_day` y `post_resume_day`.


In [ ]:
alignment = arts['event_window_alignment_summary']
quotes_links = arts['halts_quotes_link_candidates']
trades_links = arts['halts_trades_link_candidates']

display(alignment.head(30))

quotes_role = (
    quotes_links.groupby('window_role', dropna=False)
    .agg(total=('event_id_canonical', 'size'), linked=('quotes_linked', 'sum'), problem=('quotes_problem_flag', 'sum'))
    .reset_index()
)
quotes_role['linked_pct'] = 100 * quotes_role['linked'] / quotes_role['total']
quotes_role['problem_pct'] = 100 * quotes_role['problem'] / quotes_role['total']

trades_role = (
    trades_links.groupby('window_role', dropna=False)
    .agg(total=('event_id_canonical', 'size'), linked=('trades_linked', 'sum'), problem=('trades_problem_flag', 'sum'))
    .reset_index()
)
trades_role['linked_pct'] = 100 * trades_role['linked'] / trades_role['total']
trades_role['problem_pct'] = 100 * trades_role['problem'] / trades_role['total']

display(Markdown('### Cobertura por ventana: quotes'))
display(quotes_role.round(3))
display(Markdown('### Cobertura por ventana: trades'))
display(trades_role.round(3))


In [ ]:
alignment_view = (
    alignment.groupby(['window_role', 'alignment_bucket'], dropna=False)
    .agg(event_windows=('event_windows', 'sum'))
    .reset_index()
    .sort_values(['window_role', 'event_windows'], ascending=[True, False])
)
display(alignment_view)

focus_quotes = quotes_links.loc[quotes_links['quotes_problem_flag'].fillna(False)].copy()
focus_trades = trades_links.loc[trades_links['trades_problem_flag'].fillna(False)].copy()

display(Markdown('### Ejemplos linked problem: quotes'))
display(
    focus_quotes[
        ['ticker', 'window_role', 'window_date', 'event_taxonomy', 'quotes_severity', 'quotes_taxonomy', 'quotes_crossed_ratio_pct', 'quotes_file']
    ]
    .sort_values(['quotes_crossed_ratio_pct', 'ticker'], ascending=[False, True])
    .head(20)
)

display(Markdown('### Ejemplos linked problem: trades'))
display(
    focus_trades[
        ['ticker', 'window_role', 'window_date', 'event_taxonomy', 'trades_severity', 'trades_focus_issue', 'trades_issue_tokens', 'trades_file']
    ]
    .sort_values(['ticker', 'window_date'])
    .head(20)
)


## 10. Leyenda de buckets visuales

Antes de bajar a la figura intrad?a, el usuario debe interpretar el tipo de caso que est? abriendo.

- `confirmed_halt_microstructure_coherent`
  Evento de halt oficial con reacci?n microestructural coherente en `quotes` y/o `trades` cerca del halt o del reopen.
- `halt_with_quotes_signal_only`
  El halt oficial existe y `quotes` dejan se?al ?til, pero `trades` no confirman con la misma claridad.
- `halt_with_trades_signal_only`
  El halt oficial existe y `trades` dejan se?al ?til, pero `quotes` no confirman con la misma claridad.
- `halt_present_but_market_clean`
  Hay halt oficial, pero el raw visual disponible no muestra anomal?a clara en mercado.
- `market_signal_without_clear_halt_window`
  La se?al de mercado existe, pero no queda alineada de forma limpia con la ventana temporal del halt.
- `review_timestamp_alignment`
  El caso necesita revisar tiempos antes de concluir nada: timezone, fecha efectiva, `resume_trade_et` o granularidad del source.

## 11. Regla de no duplicaci?n visual

El viewer final no debe mostrar dos gr?ficos casi id?nticos del mismo evento f?sico.

Regla:

- si `halt_day` y `resume_day` caen en el mismo raw `ticker-day`, debe existir una sola figura compuesta
- esa figura debe incluir en el mismo eje temporal:
  - `halt_start_et`
  - `resume_quote_et` si existe
  - `resume_trade_et` si existe
- el ?ndice visual final debe deduplicar por `visual_key`, no por `window_role`


## 12. Viewer intrad?a deduplicado para cruce causal con `quotes` y `trades`

Estas celdas ya no trabajan sobre el join diario bruto. Trabajan sobre el ?ndice visual `<1B>` deduplicado por `visual_key`, de modo que cada opci?n corresponde a una sola vista f?sica `ticker-day` aunque ese d?a contenga varios halts.


In [ ]:
OVERLAY_SCRIPT = Path(r'C:/TSIS_Data/02_backtest_SmallCaps/notebooks/00_data_certification/auditoria/halts/cell_code/01_halts_intraday_visual_overlay.py')
overlay_mod = runpy.run_path(str(OVERLAY_SCRIPT))

causal_artifact_names = [
    'halts_lt1b_event_index',
    'halts_intraday_overlay_index',
    'halts_quotes_trades_visual_cases',
    'halts_quotes_link_candidates',
    'halts_trades_link_candidates',
    'event_window_alignment_summary',
]

causal_artifacts = {}
for name in causal_artifact_names:
    try:
        causal_artifacts[name] = load_artifact(name, CACHE_DIR)
    except FileNotFoundError:
        causal_artifacts[name] = None

status_rows = []
for name, df in causal_artifacts.items():
    status_rows.append({
        'artifact': name,
        'available': df is not None,
        'rows': None if df is None else int(len(df)),
        'columns': None if df is None else ', '.join(df.columns[:8]),
    })

display(pd.DataFrame(status_rows))

visual_cases = causal_artifacts.get('halts_quotes_trades_visual_cases')
if visual_cases is not None:
    display(Markdown('### Snapshot del ?ndice visual `<1B>` deduplicado'))
    display(
        visual_cases.groupby('visual_case_bucket', dropna=False)
        .agg(
            visual_rows=('visual_key', 'size'),
            tickers=('ticker', 'nunique'),
            total_halt_events=('events_in_visual', 'sum'),
            max_events_same_visual=('events_in_visual', 'max'),
        )
        .reset_index()
        .sort_values(['visual_rows', 'visual_case_bucket'], ascending=[False, True])
    )


In [ ]:
visual_cases = causal_artifacts.get('halts_quotes_trades_visual_cases')
if visual_cases is None or visual_cases.empty:
    display(Markdown('**halts_quotes_trades_visual_cases** no est? materializado o est? vac?o. El viewer intrad?a requiere ese artefacto.'))
else:
    viewer = overlay_mod['build_intraday_overlay_widget'](visual_cases)
    display(viewer)
